# Forecast Arena: Monte Carlo vs Chronos-2 vs TimesFM

Three models predict where a stock's **next 1-minute price** will land, as a range (10th–90th percentile):

| Model | What it is |
|---|---|
| **Monte Carlo** | Volatility + momentum model, 10,000 simulated paths from the stock's own recent surprises |
| **Chronos-2** | Amazon's 120M-parameter time-series foundation model, fine-tuned here with LoRA |
| **TimesFM 2.5** | Google's 200M-parameter time-series foundation model, fine-tuned here with LoRA |

The notebook downloads ~4 weeks of 1-minute prices for ~45 stocks, FX pairs and crypto across 15 markets, trains the AI models,
scores all three on the last 2 trading sessions (which no model sees during training), then opens a live app that
compares them minute by minute.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all** (about 20–30 minutes).

## 1. Install and write the project files

In [ ]:
!pip install -q -U yfinance flask peft "transformers>=5" "chronos-forecasting>=2.2"
# Colab ships an old torchao that the latest peft refuses to load alongside; this project doesn't use it
!pip uninstall -y -q torchao
import os
os.makedirs("/content/lab/riskmc", exist_ok=True); os.makedirs("/content/lab/web", exist_ok=True)
open("/content/lab/riskmc/__init__.py", "w").close()
os.chdir("/content/lab")

In [ ]:
%%writefile /content/lab/riskmc/markets.py
"""Exchanges, trading hours and a starter ticker list (Yahoo Finance symbols).

Hours are regular sessions in local time; exchange holidays are not modeled.
"""

from __future__ import annotations

from datetime import datetime, time, timezone
from zoneinfo import ZoneInfo

MARKETS = [
    {"id": "us", "name": "United States", "exchange": "NYSE / Nasdaq", "tz": "America/New_York",
     "sessions": [("09:30", "16:00")],
     "tickers": [("AAPL", "Apple"), ("MSFT", "Microsoft"), ("NVDA", "Nvidia"),
                 ("TSLA", "Tesla"), ("JPM", "JPMorgan Chase"), ("SPY", "S&P 500 ETF")]},
    {"id": "ca", "name": "Canada", "exchange": "TSX", "tz": "America/Toronto",
     "sessions": [("09:30", "16:00")],
     "tickers": [("SHOP.TO", "Shopify"), ("RY.TO", "Royal Bank of Canada"), ("ENB.TO", "Enbridge")]},
    {"id": "br", "name": "Brazil", "exchange": "B3", "tz": "America/Sao_Paulo",
     "sessions": [("10:00", "17:00")],
     "tickers": [("PETR4.SA", "Petrobras"), ("VALE3.SA", "Vale"), ("ITUB4.SA", "Itaú Unibanco")]},
    {"id": "uk", "name": "United Kingdom", "exchange": "LSE", "tz": "Europe/London",
     "sessions": [("08:00", "16:30")],
     "tickers": [("SHEL.L", "Shell"), ("AZN.L", "AstraZeneca"), ("HSBA.L", "HSBC"), ("BP.L", "BP")]},
    {"id": "de", "name": "Germany", "exchange": "Xetra", "tz": "Europe/Berlin",
     "sessions": [("09:00", "17:30")],
     "tickers": [("SAP.DE", "SAP"), ("SIE.DE", "Siemens"), ("ALV.DE", "Allianz")]},
    {"id": "fr", "name": "France", "exchange": "Euronext Paris", "tz": "Europe/Paris",
     "sessions": [("09:00", "17:30")],
     "tickers": [("MC.PA", "LVMH"), ("TTE.PA", "TotalEnergies"), ("AIR.PA", "Airbus")]},
    {"id": "ch", "name": "Switzerland", "exchange": "SIX", "tz": "Europe/Zurich",
     "sessions": [("09:00", "17:30")],
     "tickers": [("NESN.SW", "Nestlé"), ("NOVN.SW", "Novartis"), ("UBSG.SW", "UBS Group")]},
    {"id": "in", "name": "India", "exchange": "NSE", "tz": "Asia/Kolkata",
     "sessions": [("09:15", "15:30")],
     "tickers": [("RELIANCE.NS", "Reliance Industries"), ("TCS.NS", "Tata Consultancy"),
                 ("HDFCBANK.NS", "HDFC Bank"), ("INFY.NS", "Infosys")]},
    {"id": "sg", "name": "Singapore", "exchange": "SGX", "tz": "Asia/Singapore",
     "sessions": [("09:00", "12:00"), ("13:00", "17:00")],
     "tickers": [("D05.SI", "DBS Group"), ("O39.SI", "OCBC Bank")]},
    {"id": "hk", "name": "Hong Kong", "exchange": "HKEX", "tz": "Asia/Hong_Kong",
     "sessions": [("09:30", "12:00"), ("13:00", "16:00")],
     "tickers": [("0700.HK", "Tencent"), ("9988.HK", "Alibaba"), ("0005.HK", "HSBC Holdings")]},
    {"id": "kr", "name": "South Korea", "exchange": "KRX", "tz": "Asia/Seoul",
     "sessions": [("09:00", "15:30")],
     "tickers": [("005930.KS", "Samsung Electronics"), ("000660.KS", "SK Hynix")]},
    {"id": "jp", "name": "Japan", "exchange": "Tokyo SE", "tz": "Asia/Tokyo",
     "sessions": [("09:00", "11:30"), ("12:30", "15:30")],
     "tickers": [("7203.T", "Toyota"), ("6758.T", "Sony"), ("9984.T", "SoftBank Group")]},
    {"id": "au", "name": "Australia", "exchange": "ASX", "tz": "Australia/Sydney",
     "sessions": [("10:00", "16:00")],
     "tickers": [("BHP.AX", "BHP Group"), ("CBA.AX", "Commonwealth Bank"), ("CSL.AX", "CSL")]},
    {"id": "fx", "name": "Currencies", "exchange": "FX, 24/5", "tz": "America/New_York",
     "sessions": "fx",
     "tickers": [("EURUSD=X", "Euro / US Dollar"), ("USDJPY=X", "US Dollar / Yen"),
                 ("GBPUSD=X", "Pound / US Dollar")]},
    {"id": "crypto", "name": "Crypto", "exchange": "24/7", "tz": "UTC",
     "sessions": "always",
     "tickers": [("BTC-USD", "Bitcoin"), ("ETH-USD", "Ethereum"), ("SOL-USD", "Solana")]},
]


def _t(s: str) -> time:
    return time.fromisoformat(s)


def status(market: dict, now: datetime | None = None) -> dict:
    now = now or datetime.now(timezone.utc)
    local = now.astimezone(ZoneInfo(market["tz"]))
    sessions = market["sessions"]
    if sessions == "always":
        state = "open"
    elif sessions == "fx":
        # Sunday 17:00 to Friday 17:00 New York time
        wd, t = local.weekday(), local.time()
        closed = wd == 5 or (wd == 4 and t >= _t("17:00")) or (wd == 6 and t < _t("17:00"))
        state = "closed" if closed else "open"
    elif local.weekday() >= 5:
        state = "closed"
    else:
        t = local.time()
        if any(_t(a) <= t < _t(b) for a, b in sessions):
            state = "open"
        elif _t(sessions[0][0]) <= t < _t(sessions[-1][1]):
            state = "break"
        else:
            state = "closed"
    hours = "24/7" if sessions == "always" else "24/5" if sessions == "fx" else \
        ", ".join(f"{a}–{b}" for a, b in sessions)
    return {"state": state, "local_time": local.strftime("%a %H:%M"), "hours": hours}


def catalog() -> list[dict]:
    rows = []
    for m in MARKETS:
        rows.append({
            "id": m["id"], "name": m["name"], "exchange": m["exchange"], **status(m),
            "tickers": [{"symbol": s, "name": n} for s, n in m["tickers"]],
        })
    order = {"open": 0, "break": 1, "closed": 2}
    return sorted(rows, key=lambda r: order[r["state"]])


def market_for(symbol: str) -> dict | None:
    for m in MARKETS:
        if any(s == symbol for s, _ in m["tickers"]):
            return m
    return None

In [ ]:
%%writefile /content/lab/riskmc/minute.py
"""Next-minute Monte Carlo forecaster and the scoring shared by every model.

Model (filtered historical simulation), using only information up to the current bar:
    r_next = phi * r_last + sigma * z
- sigma: EWMA volatility (lambda = 0.97), reacts quickly to bursts
- phi:   shrunk rolling AR(1) coefficient, captures short-term momentum / bid-ask bounce
- z:     drawn from this ticker's own recent standardized forecast errors, so the
         band width and tail shape calibrate themselves per asset
The forecast distribution comes from simulating many next-minute prices.

Every model in the app reports the same nine quantiles (10th..90th percentile),
which is what TimesFM produces natively, so the comparison is like for like.
"""

from __future__ import annotations

import numpy as np
import pandas as pd

MAX_GAP = pd.Timedelta(minutes=5)   # returns across longer gaps (overnight, lunch) are skipped
LEVELS = np.round(np.arange(0.1, 0.91, 0.1), 2)             # 0.1, 0.2, ..., 0.9
QCOLS = [f"q{int(round(l * 100))}" for l in LEVELS]          # q10 ... q90


def minute_returns(closes: pd.Series) -> pd.Series:
    """Log returns between consecutive bars; NaN where the bars are separated by a gap."""
    r = np.log(closes / closes.shift(1))
    gap = closes.index.to_series().diff() > MAX_GAP
    return r.mask(gap.to_numpy())


def returns_bps(closes: pd.Series) -> pd.Series:
    """1-minute log returns in basis points, with returns across market gaps dropped.

    This is what the AI models read in "returns" mode: each window is then rescaled by its own
    recent volatility, instead of by a price level that drifts far more than one minute moves.
    """
    return (minute_returns(closes) * 1e4).dropna()


def model_frame(closes: pd.Series, lookback: int = 120, lam: float = 0.97,
                shrink: float = 0.5) -> pd.DataFrame:
    """Per-bar model state. Row t only uses returns up to and including bar t."""
    r = minute_returns(closes)
    sigma = np.sqrt(r.pow(2).ewm(alpha=1 - lam, adjust=False, ignore_na=True,
                                 min_periods=20).mean()).clip(lower=1e-6)
    r_prev = r.shift(1)
    num = (r * r_prev).rolling(lookback, min_periods=30).sum()
    den = (r_prev ** 2).rolling(lookback, min_periods=30).sum()
    phi = (shrink * num / den).clip(-0.3, 0.3).fillna(0.0)
    mean_next = phi * r.fillna(0.0)                       # expected return of bar t+1
    z = (r - mean_next.shift(1)) / sigma.shift(1)         # realized surprise at bar t
    return pd.DataFrame({"r": r, "sigma": sigma, "phi": phi, "mean_next": mean_next, "z": z})


def mc_quantiles(closes: pd.Series, frame: pd.DataFrame, i: int, calib: int = 300,
                 n_paths: int = 10_000, rng: np.random.Generator | None = None) -> np.ndarray | None:
    """Monte Carlo quantiles (LEVELS) of the close of bar i+1, using bars up to i."""
    rng = rng or np.random.default_rng()
    zs = frame["z"].iloc[:i + 1].dropna().to_numpy()[-calib:]
    sigma, mean = frame["sigma"].iloc[i], frame["mean_next"].iloc[i]
    if len(zs) < 60 or np.isnan(sigma):
        return None
    sim_r = mean + sigma * rng.choice(zs, n_paths)        # Monte Carlo over past surprises
    return float(closes.iloc[i]) * np.exp(np.quantile(sim_r, LEVELS))


def rescale(q: np.ndarray, scale: float) -> np.ndarray:
    """Widen (scale > 1) or narrow a quantile forecast around its median."""
    q = np.asarray(q, dtype=float)
    med = q[..., [len(LEVELS) // 2]]
    return med + scale * (q - med)


def pinball_bps(q: np.ndarray, actual: np.ndarray, last: np.ndarray) -> np.ndarray:
    """Average quantile (pinball) loss over LEVELS, in basis points of the last price.

    A proper scoring rule: it rewards ranges that are both correctly placed and tight,
    so it is the single number used to rank models.
    """
    q, actual, last = np.atleast_2d(q), np.asarray(actual)[:, None], np.asarray(last)[:, None]
    diff = actual - q
    loss = np.maximum(LEVELS * diff, (LEVELS - 1) * diff)
    return (loss / last * 1e4).mean(axis=1)


def fit_scale(q: np.ndarray, actual: np.ndarray, last: np.ndarray) -> float:
    """Spread multiplier that minimizes pinball loss on a calibration set."""
    grid = np.round(np.arange(0.4, 3.01, 0.05), 2)
    losses = [pinball_bps(rescale(q, s), actual, last).mean() for s in grid]
    return float(grid[int(np.argmin(losses))])


# ---- Game scoring -------------------------------------------------------------------------
# Each round every model gets the same outer-range width (the average of the models' own
# 10th-90th percentile widths), centered on its own median. The inner range is 20% of that width.
# A price in the inner range scores 2, in the outer range 1, otherwise 0, so the game only tests
# where each model aims, not who draws the widest range.
INNER_SHARE = 0.2


def game_ranges(center, width):
    """(outer_lo, outer_hi, inner_lo, inner_hi) for a range of `width` centered on `center`."""
    center, width = np.asarray(center, dtype=float), np.asarray(width, dtype=float)
    return center - width / 2, center + width / 2, center - INNER_SHARE * width / 2, center + INNER_SHARE * width / 2


def points(center, width, actual):
    """2 inside the inner range, 1 inside the outer range, 0 otherwise."""
    off = np.abs(np.asarray(actual, dtype=float) - np.asarray(center, dtype=float))
    width = np.asarray(width, dtype=float)
    return np.where(off <= INNER_SHARE * width / 2, 2, np.where(off <= width / 2, 1, 0))


def add_game_points(rows: pd.DataFrame, by) -> pd.DataFrame:
    """Score every model's forecasts with the shared width: the mean width of all models' forecasts
    for the same target (grouped by `by`, e.g. ["symbol", "time"])."""
    rows = rows.copy()
    rows["game_width"] = (rows["q90"] - rows["q10"]).groupby([rows[c] for c in by]).transform("mean")
    rows["points"] = points(rows["q50"], rows["game_width"], rows["actual"])
    return rows


def score(rows: pd.DataFrame) -> dict:
    """Scores for a set of forecasts with columns q10..q90, last, actual."""
    if rows is None or rows.empty:
        return {"n": 0}
    q = rows[QCOLS].to_numpy()
    last, actual = rows["last"].to_numpy(), rows["actual"].to_numpy()
    med = rows["q50"].to_numpy()
    moved = (actual != last) & (med != last)
    direction = (np.sign(actual - last) == np.sign(med - last))[moved]
    err_model = np.abs(actual - med) / last * 1e4
    err_naive = np.abs(actual - last) / last * 1e4
    pts = rows["points"].to_numpy() if "points" in rows else points(med, rows["q90"] - rows["q10"], actual)
    return {
        "n": int(len(rows)),
        "points": int(pts.sum()),
        "points_per_round": float(pts.mean()),
        "bullseye": float((pts == 2).mean()),
        "coverage80": float(((actual >= rows["q10"]) & (actual <= rows["q90"])).mean()),
        "width80_bps": float(((rows["q90"] - rows["q10"]) / last * 1e4).mean()),
        "pinball_bps": float(pinball_bps(q, actual, last).mean()),
        "direction_acc": float(direction.mean()) if len(direction) else None,
        "mae_bps": float(err_model.mean()),
        "naive_mae_bps": float(err_naive.mean()),
    }

In [ ]:
%%writefile /content/lab/riskmc/forecasters.py
"""The three competing next-minute forecasters behind one interface.

Each forecaster maps a list of price histories to a (batch, 9) array of price
quantiles at LEVELS (10th..90th percentile) for the next bar. A learned spread
`scale` (see minute.fit_scale) is applied on top of every model's raw output.

- Monte Carlo: filtered historical simulation (riskmc.minute)
- Chronos-2:   Amazon's time-series foundation model (chronos-forecasting, Apache 2.0)
- TimesFM 2.5: Google's time-series foundation model (Hugging Face Transformers port, Apache 2.0)
"""

from __future__ import annotations

import inspect
from pathlib import Path

import numpy as np
import pandas as pd

from .minute import LEVELS, mc_quantiles, model_frame, rescale, returns_bps

NAN_ROW = np.full(len(LEVELS), np.nan)


class Forecaster:
    key = "base"
    name = "Base"
    blurb = ""

    def __init__(self):
        self.scale = 1.0
        self.available = True
        self.error = None

    def raw_quantiles(self, histories: list[pd.Series]) -> np.ndarray:
        raise NotImplementedError

    def quantiles(self, histories: list[pd.Series]) -> np.ndarray:
        return rescale(self.raw_quantiles(histories), self.scale)

    def replay(self, closes: pd.Series, idxs: list[int]) -> np.ndarray:
        """Quantiles for bar i+1 for each i in idxs, using only bars up to i."""
        return self.quantiles([closes.iloc[:i + 1] for i in idxs])

    def info(self) -> dict:
        return {"key": self.key, "name": self.name, "blurb": self.blurb, "available": self.available,
                "error": self.error, "scale": self.scale, "fine_tuned": getattr(self, "fine_tuned", False)}


class MonteCarloForecaster(Forecaster):
    key, name = "mc", "Monte Carlo"
    blurb = "Volatility + momentum model, 10,000 simulated paths from the stock's own recent surprises"

    def __init__(self, n_paths: int = 10_000, seed: int | None = None):
        super().__init__()
        self.n_paths, self.rng = n_paths, np.random.default_rng(seed)

    def raw_quantiles(self, histories):
        out = []
        for h in histories:
            q = mc_quantiles(h, model_frame(h), len(h) - 1, n_paths=self.n_paths, rng=self.rng)
            out.append(NAN_ROW if q is None else q)
        return np.array(out)

    def replay(self, closes, idxs):
        frame = model_frame(closes)                       # row i only depends on bars <= i
        out = []
        for i in idxs:
            q = mc_quantiles(closes, frame, i, n_paths=self.n_paths, rng=self.rng)
            out.append(NAN_ROW if q is None else q)
        return rescale(np.array(out), self.scale)


def _batches(items, size):
    for k in range(0, len(items), size):
        yield items[k:k + size]


class AIForecaster(Forecaster):
    """Shared input handling for the foundation models.

    input_mode "prices":  the model reads the last `context` prices and forecasts the next price.
    input_mode "returns": the model reads the last `context` 1-minute returns (bps, gaps dropped) and
                          forecasts the next return, converted back to a price range.
    """
    input_mode = "returns"
    context = 512

    def _contexts(self, histories: list[pd.Series]) -> list[np.ndarray]:
        if self.input_mode == "prices":
            return [h.to_numpy(dtype=np.float32)[-self.context:] for h in histories]
        return [returns_bps(h).to_numpy(dtype=np.float32)[-self.context:] for h in histories]

    def _to_prices(self, q: np.ndarray, histories: list[pd.Series]) -> np.ndarray:
        if self.input_mode == "prices":
            return q
        last = np.array([float(h.iloc[-1]) for h in histories])[:, None]
        return last * np.exp(q / 1e4)

    def _predict(self, arrays: list[np.ndarray]) -> np.ndarray:
        raise NotImplementedError

    def raw_quantiles(self, histories):
        arrays = self._contexts(histories)
        q = np.concatenate([self._predict(b) for b in _batches(arrays, self.batch_size)])
        return self._to_prices(q, histories)

    def info(self) -> dict:
        return {**super().info(), "input_mode": self.input_mode, "context": self.context}


class Chronos2Forecaster(AIForecaster):
    key, name = "chronos2", "Chronos-2"
    blurb = "Amazon's 120M-parameter time-series foundation model"

    def __init__(self, path: str = "amazon/chronos-2", device: str = "cpu", context: int = 512,
                 batch_size: int = 128):
        """`path` is a Hugging Face model id or a fine-tuned checkpoint folder (LoRA adapters load directly)."""
        super().__init__()
        from chronos import BaseChronosPipeline

        self.path, self.device, self.context, self.batch_size = path, device, context, batch_size
        self.pipe = BaseChronosPipeline.from_pretrained(path, device_map=device)
        self.fine_tuned = path != "amazon/chronos-2"

    def _predict(self, arrays: list[np.ndarray]) -> np.ndarray:
        import torch

        out = self.pipe.predict_quantiles([torch.tensor(a) for a in arrays], prediction_length=1,
                                          quantile_levels=LEVELS.tolist())
        q = out[0] if isinstance(out, tuple) else out
        if isinstance(q, (list, tuple)):                  # Chronos-2: one tensor per series
            return np.stack([np.asarray(x.float().cpu()).reshape(-1)[:len(LEVELS)] for x in q])
        return np.asarray(q.float().cpu()).reshape(len(arrays), -1)[:, :len(LEVELS)]

    def fine_tune(self, series: list[np.ndarray], out_dir: str, validation: list[np.ndarray] | None = None,
                  num_steps: int = 600, learning_rate: float = 1e-4, batch_size: int = 64, mode: str = "lora",
                  every: int = 25, min_past: int = 64) -> None:
        """Fine-tune with chronos-forecasting's own `fit` (LoRA by default).

        With `validation` windows, the trainer scores held-out loss before step 1 and every `every`
        steps, and keeps the checkpoint with the lowest validation loss.
        """
        if not hasattr(self.pipe, "fit"):
            raise RuntimeError("This chronos-forecasting version has no fit(); upgrade with "
                               "pip install -U chronos-forecasting")
        wanted = {"inputs": [s.astype(np.float32) for s in series], "prediction_length": 1,
                  "validation_inputs": [v.astype(np.float32) for v in validation] if validation else None,
                  "finetune_mode": mode, "num_steps": num_steps, "learning_rate": learning_rate,
                  "batch_size": batch_size, "context_length": self.context, "min_past": min_past,
                  "output_dir": out_dir}
        params = inspect.signature(self.pipe.fit).parameters
        kwargs = {k: v for k, v in wanted.items() if k in params}
        if any(p.kind == p.VAR_KEYWORD for p in params.values()):      # forwarded to TrainingArguments
            kwargs.update(logging_steps=every)
            if validation:
                kwargs.update(eval_steps=every, save_steps=every, eval_on_start=True)
        print("Chronos-2 fit() with", {k: (f"{len(v)} series" if isinstance(v, list) else v) for k, v in kwargs.items()})
        self.pipe = self.pipe.fit(**kwargs)                # fit() saves to out_dir/finetuned-ckpt itself
        self.path, self.fine_tuned = str(Path(out_dir) / "finetuned-ckpt"), True


class TimesFMForecaster(AIForecaster):
    key, name = "timesfm", "TimesFM 2.5"
    blurb = "Google's 200M-parameter time-series foundation model"

    def __init__(self, path: str = "google/timesfm-2.5-200m-transformers", device: str = "cpu",
                 context: int = 512, batch_size: int = 128, adapter: str | None = None):
        """`adapter` is a folder with LoRA weights saved by `fine_tune`, applied on top of the base model."""
        super().__init__()
        from transformers import TimesFm2_5ModelForPrediction

        self.device, self.context, self.batch_size = device, context, batch_size
        self.model = TimesFm2_5ModelForPrediction.from_pretrained(path, torch_dtype="float32").to(device)
        self.fine_tuned = adapter is not None
        if adapter:
            from peft import PeftModel
            self.model = PeftModel.from_pretrained(self.model, adapter).to(device)
        self.model.eval()

    def _predict(self, arrays):
        import torch

        with torch.no_grad():
            res = self.model(past_values=[torch.tensor(a, device=self.device) for a in arrays],
                             forecast_context_len=self.context, truncate_negative=False)
        # full_predictions: (batch, horizon, 10) = mean, then the 10th..90th percentiles
        return res.full_predictions[:, 0, 1:1 + len(LEVELS)].float().cpu().numpy()

    def fine_tune(self, series: list[np.ndarray], out_dir: str, validation: list[np.ndarray] | None = None,
                  targets: list[np.ndarray] | None = None, num_steps: int = 600, learning_rate: float = 5e-5,
                  batch_size: int = 64, lora_r: int = 4, every: int = 25, seed: int = 0) -> list[dict]:
        """LoRA fine-tuning, following TimesFM's own example (timesfm-forecasting/examples/finetuning).

        Each step samples random (512 context -> next price) windows from the training series and
        minimizes the model's built-in loss (MSE + quantile loss in normalized space). With
        `validation` windows, held-out loss is measured before step 1 and every `every` steps, and
        the adapter weights with the lowest validation loss are kept. `targets` (from
        training.finetune_targets) restricts training to targets that don't follow a market gap.
        Returns the loss history.
        """
        import torch
        from peft import LoraConfig, get_peft_model

        rng = np.random.default_rng(seed)
        if targets is None:
            targets = [np.arange(self.context, len(s)) for s in series]
        pool = [(s, t) for s, t in zip(series, targets) if len(t)]
        model = get_peft_model(self.model, LoraConfig(r=lora_r, lora_alpha=2 * lora_r, target_modules="all-linear",
                                                      lora_dropout=0.05, bias="none"))
        model.print_trainable_parameters()
        opt = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_steps)

        def val_loss() -> float | None:
            if not validation:
                return None
            model.eval()
            total, n = 0.0, 0
            with torch.no_grad():
                for b in _batches(validation, 64):
                    ctx = [torch.tensor(w[:-1], device=self.device) for w in b]
                    tgt = torch.tensor([[w[-1]] for w in b], device=self.device)
                    total += model(past_values=ctx, future_values=tgt, forecast_context_len=self.context,
                                   truncate_negative=False).loss.item() * len(b)
                    n += len(b)
            model.train()
            return total / n

        lora_state = lambda: {k: v.detach().clone() for k, v in model.state_dict().items() if "lora_" in k}
        history = [{"step": 0, "train": None, "val": val_loss()}]
        best_val, best_state, best_step = history[0]["val"], lora_state(), 0
        print(f"  TimesFM step 0    validation loss {best_val}")
        recent = []
        model.train()
        for step in range(1, num_steps + 1):
            ctx, tgt = [], []
            for _ in range(batch_size):
                s, t = pool[rng.integers(len(pool))]
                i = int(t[rng.integers(len(t))])                  # target position
                ctx.append(torch.tensor(s[i - self.context:i], device=self.device))
                tgt.append(float(s[i]))
            loss = model(past_values=ctx, future_values=torch.tensor(tgt, device=self.device)[:, None],
                         forecast_context_len=self.context, truncate_negative=False).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); opt.zero_grad(); sched.step()
            recent.append(loss.item())
            if step % every == 0 or step == num_steps:
                v = val_loss()
                history.append({"step": step, "train": float(np.mean(recent)), "val": v})
                recent = []
                mark = ""
                if v is not None and (best_val is None or v < best_val):
                    best_val, best_state, best_step, mark = v, lora_state(), step, "  ← best"
                print(f"  TimesFM step {step:<4} train loss {history[-1]['train']:.5f}"
                      + (f"  validation loss {v:.5f}{mark}" if v is not None else ""))
        if validation:
            model.load_state_dict(best_state, strict=False)
            print(f"  Kept the adapter from step {best_step} (lowest validation loss {best_val:.5f})")
        model.eval()
        model.save_pretrained(out_dir)
        self.model, self.fine_tuned = model, True
        return history


def build(load_ai: bool = True, device: str | None = None, chronos_path: str | None = None,
          timesfm_adapter: str | None = None, scales: dict | None = None,
          configs: dict | None = None) -> dict[str, Forecaster]:
    """All forecasters that can be loaded here. A model that fails to load stays listed as unavailable."""
    if device is None:
        try:
            import torch
            device = "cuda" if torch.cuda.is_available() else "cpu"
        except ImportError:
            device = "cpu"
    models: dict[str, Forecaster] = {"mc": MonteCarloForecaster()}
    specs = [("chronos2", lambda: Chronos2Forecaster(chronos_path or "amazon/chronos-2", device)),
             ("timesfm", lambda: TimesFMForecaster(device=device, adapter=timesfm_adapter))]
    for key, make in specs:
        if not load_ai:
            continue
        try:
            models[key] = make()
        except Exception as e:                            # keep the app usable without this model
            stub = Forecaster()
            stub.key = key
            stub.name = {"chronos2": Chronos2Forecaster.name, "timesfm": TimesFMForecaster.name}[key]
            stub.available, stub.error = False, f"{type(e).__name__}: {e}"
            models[key] = stub
            print(f"Could not load {stub.name}: {stub.error}")
    for key, cfg in (configs or {}).items():
        if key in models and isinstance(models[key], AIForecaster):
            models[key].input_mode, models[key].context = cfg["input_mode"], int(cfg["context"])
    for key, s in (scales or {}).items():
        if key in models:
            models[key].scale = float(s)
    return models

In [ ]:
%%writefile /content/lab/riskmc/training.py
"""Download minute history, train each forecaster, and score them on unseen days.

Time is split three ways per ticker, so nothing is scored on data it learned from:
    [ fine-tune ............ ][ calibrate: 1 session ][ test: last 2 sessions ]
- fine-tune: Chronos-2 is fine-tuned on these prices
- calibrate: each model's range width (spread scale) is fitted here
- test:      all three models are scored here, out of sample
"""

from __future__ import annotations

import json
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

from .markets import MARKETS
from .minute import LEVELS, MAX_GAP, QCOLS, add_game_points, fit_scale, pinball_bps, rescale, returns_bps, score

TRAIN_TICKERS = [s for m in MARKETS for s, _ in m["tickers"]]


def download_minutes(symbol: str, days: int = 28, retries: int = 3) -> pd.Series:
    """Completed 1-minute closes for the last `days` days (Yahoo serves 7 days per request)."""
    import yfinance as yf

    end = pd.Timestamp.now(tz="UTC").normalize() + pd.Timedelta(days=1)
    parts = []
    for k in range(int(np.ceil(days / 7))):
        e, s = end - pd.Timedelta(days=7 * k), end - pd.Timedelta(days=7 * (k + 1))
        for attempt in range(retries):
            try:
                h = yf.Ticker(symbol).history(start=s.strftime("%Y-%m-%d"), end=e.strftime("%Y-%m-%d"),
                                              interval="1m", auto_adjust=True)
                break
            except Exception:
                time.sleep(2 * (attempt + 1))
        else:
            continue
        if not h.empty:
            parts.append(h["Close"])
    if not parts:
        return pd.Series(dtype=float)
    closes = pd.concat(parts).sort_index()
    closes = closes[~closes.index.duplicated()].dropna()
    closes = closes[closes.index.second == 0]
    return closes[closes.index + pd.Timedelta(minutes=1) <= pd.Timestamp.now(tz=closes.index.tz)]


def download_all(tickers=TRAIN_TICKERS, days: int = 28, min_bars: int = 2_000) -> dict[str, pd.Series]:
    data = {}
    for i, t in enumerate(tickers, 1):
        s = download_minutes(t, days)
        status = f"{len(s):,} bars" if len(s) >= min_bars else f"skipped ({len(s)} bars)"
        print(f"[{i}/{len(tickers)}] {t:<12} {status}")
        if len(s) >= min_bars:
            data[t] = s
    return data


def split_points(closes: pd.Series, test_sessions: int = 2, calib_sessions: int = 1):
    """Timestamps where the calibration and test windows start (by trading session)."""
    sessions = pd.Index(closes.index.normalize().unique())
    if len(sessions) < test_sessions + calib_sessions + 3:
        return None
    test_start = sessions[-test_sessions]
    calib_start = sessions[-(test_sessions + calib_sessions)]
    return calib_start, test_start


def eval_indices(closes: pd.Series, start, end, n: int, min_history: int = 600) -> list[int]:
    """Up to n evenly spaced bars i whose next bar (the target) falls in [start, end)."""
    t = closes.index
    ok = [i for i in range(min_history, len(closes) - 1)
          if start <= t[i + 1] < end and t[i + 1] - t[i] <= MAX_GAP]
    if len(ok) > n:
        ok = [ok[k] for k in np.linspace(0, len(ok) - 1, n).astype(int)]
    return ok


def collect(models: dict, data: dict[str, pd.Series], window: str, n_per_ticker: int,
            verbose: bool = True) -> pd.DataFrame:
    """Forecast every sampled point in the given window with every model."""
    rows = []
    for sym, closes in data.items():
        sp = split_points(closes)
        if sp is None:
            continue
        calib_start, test_start = sp
        start, end = (calib_start, test_start) if window == "calib" else (test_start, closes.index[-1] + MAX_GAP)
        idxs = eval_indices(closes, start, end, n_per_ticker)
        if not idxs:
            continue
        last = closes.to_numpy()[idxs]
        actual = closes.to_numpy()[[i + 1 for i in idxs]]
        for key, m in models.items():
            if not m.available:
                continue
            q = m.replay(closes, idxs)
            df = pd.DataFrame(q, columns=QCOLS)
            df["model"], df["symbol"], df["last"], df["actual"] = key, sym, last, actual
            df["time"] = closes.index[[i + 1 for i in idxs]]
            rows.append(df.dropna(subset=QCOLS))
        if verbose:
            print(f"  {window}: {sym:<12} {len(idxs)} points")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def calibrate(models: dict, calib_rows: pd.DataFrame) -> dict[str, float]:
    """Fit each model's spread scale on the calibration window (raw outputs, scale = 1)."""
    scales = {}
    for key, m in models.items():
        r = calib_rows[calib_rows["model"] == key]
        if r.empty:
            continue
        m.scale = fit_scale(r[QCOLS].to_numpy(), r["actual"].to_numpy(), r["last"].to_numpy())
        scales[key] = m.scale
        print(f"  {m.name:<12} spread scale = {m.scale:.2f}")
    return scales


def summarize(models: dict, rows: pd.DataFrame) -> pd.DataFrame:
    rows = add_game_points(rows, ["symbol", "time"])       # same outer width for every model, per target
    out = []
    for key, m in models.items():
        r = rows[rows["model"] == key]
        if r.empty:
            continue
        s = score(r)
        out.append({"model": m.name, "forecasts": s["n"], "points per round": s["points_per_round"],
                    "bullseye rate": s["bullseye"], "in 8-in-10 range": s["coverage80"],
                    "avg range width (bps)": s["width80_bps"], "pinball loss (bps)": s["pinball_bps"],
                    "direction right": s["direction_acc"]})
    return pd.DataFrame(out).set_index("model").sort_values("pinball loss (bps)")


def save_report(path: str | Path, models: dict, zero_shot: pd.DataFrame, trained: pd.DataFrame,
                scales: dict, data: dict, extra: dict | None = None) -> dict:
    report = {
        "trained_at": datetime.now(timezone.utc).isoformat(timespec="minutes"),
        "tickers": len(data),
        "bars": int(sum(len(s) for s in data.values())),
        "scales": scales,
        "zero_shot": zero_shot.reset_index().to_dict(orient="records"),
        "configs": {k: {"input_mode": m.input_mode, "context": m.context}
                    for k, m in models.items() if hasattr(m, "input_mode")},
        "trained": trained.reset_index().to_dict(orient="records"),
        **(extra or {}),
    }
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(report, indent=2, default=str))
    return report


def validation_windows(data: dict[str, pd.Series], per_ticker: int = 20, context: int = 512,
                       mode: str = "prices", seed: int = 0) -> list[np.ndarray]:
    """Held-out windows from the calibration session: `context` values followed by the target value.

    `mode` matches the model's input: prices, or 1-minute returns in bps. Used to watch validation
    loss while fine-tuning; none of these targets are in the training series.
    """
    rng = np.random.default_rng(seed)
    out = []
    for closes in data.values():
        sp = split_points(closes)
        if sp is None:
            continue
        if mode == "returns":
            r = returns_bps(closes)
            ok = np.flatnonzero((r.index >= sp[0]) & (r.index < sp[1]))
            ok = ok[ok >= context]
            for j in rng.choice(ok, size=min(per_ticker, len(ok)), replace=False):
                out.append(r.to_numpy(dtype=np.float32)[j - context:j + 1])
            continue
        t = closes.index
        ok = [i for i in range(context, len(closes))
              if sp[0] <= t[i] < sp[1] and t[i] - t[i - 1] <= MAX_GAP]
        for i in rng.choice(ok, size=min(per_ticker, len(ok)), replace=False):
            out.append(closes.to_numpy(dtype=np.float32)[i - context:i + 1])
    return out


def _pretrain(closes: pd.Series) -> pd.Series | None:
    sp = split_points(closes)
    return None if sp is None else closes[closes.index < sp[0]]


def finetune_series(data: dict[str, pd.Series], split_sessions: bool = False,
                    min_len: int = 65, mode: str = "prices") -> list[np.ndarray]:
    """Per-ticker histories that end before the calibration window.

    mode "returns": 1-minute returns in bps with gap returns dropped, one series per ticker.
    mode "prices" with `split_sessions`: prices cut at overnight / lunch gaps, so no training target
    is a next-morning opening price (a jump no 1-minute model should be fitted to).
    """
    out = []
    for closes in data.values():
        c = _pretrain(closes)
        if c is None:
            continue
        if mode == "returns":
            out.append(returns_bps(closes)[lambda r: r.index <= c.index[-1]].to_numpy(dtype=np.float32))
            continue
        if not split_sessions:
            out.append(c.to_numpy(dtype=np.float32))
            continue
        session_id = (c.index.to_series().diff() > MAX_GAP).cumsum().to_numpy()
        for k in np.unique(session_id):
            seg = c.to_numpy(dtype=np.float32)[session_id == k]
            if len(seg) >= min_len:
                out.append(seg)
    return out


def finetune_targets(data: dict[str, pd.Series], context: int = 512) -> list[np.ndarray]:
    """For each series from finetune_series(data), the positions that are valid prediction targets:
    a full `context` of history before them and no gap between the target and the bar before it."""
    out = []
    for closes in data.values():
        c = _pretrain(closes)
        if c is None:
            continue
        ok = np.ones(len(c), dtype=bool)
        ok[:context] = False
        ok[1:] &= (c.index[1:] - c.index[:-1]) <= MAX_GAP
        out.append(np.flatnonzero(ok))
    return out


def select_config(key: str, model, data: dict[str, pd.Series], modes=("prices", "returns"),
                  contexts=(128, 512), n_per_ticker: int = 60) -> pd.DataFrame:
    """Try each input mode and context length on the calibration session; keep the best.

    Configurations are compared by pinball loss after each gets its best range width, so the choice
    reflects where the ranges sit and how they adapt, not just their starting width. The test
    sessions are never used here.
    """
    rows = []
    for mode in modes:
        for ctx in contexts:
            model.input_mode, model.context, model.scale = mode, ctx, 1.0
            r = collect({key: model}, data, "calib", n_per_ticker, verbose=False)
            q, actual, last = r[QCOLS].to_numpy(), r["actual"].to_numpy(), r["last"].to_numpy()
            sc = fit_scale(q, actual, last)
            rows.append({"input": mode, "context": ctx, "pinball (bps)": float(pinball_bps(rescale(q, sc), actual, last).mean()),
                         "raw coverage 80%": float(((actual >= r["q10"]) & (actual <= r["q90"])).mean())})
    table = pd.DataFrame(rows).sort_values("pinball (bps)").reset_index(drop=True)
    model.input_mode, model.context = table.loc[0, "input"], int(table.loc[0, "context"])
    print(f"  {model.name}: using {model.input_mode} input, context {model.context}")
    return table

In [ ]:
%%writefile /content/lab/app.py
"""Forecast Arena: Monte Carlo vs Chronos-2 vs TimesFM, scored live on real minute prices.

    python app.py              # local, Monte Carlo only (AI models load if installed)
    In Colab: see colab/Forecast_Arena.ipynb, which calls serve(models, report)
"""

from __future__ import annotations

import json
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from flask import Flask, jsonify, send_from_directory

from riskmc.markets import catalog, market_for, status
from riskmc.minute import MAX_GAP, QCOLS, add_game_points, game_ranges, points, score

ROOT = Path(__file__).resolve().parent
app = Flask(__name__, static_folder=str(ROOT / "web"))

CACHE_SECONDS = 5
REPLAY_BARS = 120
MODELS: dict = {}
REPORT: dict | None = None

_lock = threading.Lock()
_model_lock = threading.Lock()                               # one model call at a time (shared GPU)
_bars: dict[str, tuple[float, pd.Series, dict]] = {}       # symbol -> (fetched_at, closes, meta)
_replays: dict[str, tuple[pd.Timestamp, pd.DataFrame]] = {}
_live: dict[str, dict[pd.Timestamp, dict]] = {}           # symbol -> base bar -> {made_at, last, q: {model: array}}
_feed: dict[str, dict] = {}                               # symbol -> newest bar seen + feed latency


def get_bars(symbol: str) -> tuple[pd.Series, dict]:
    """Completed 1-minute closes for the last few sessions (cached briefly)."""
    with _lock:
        hit = _bars.get(symbol)
    if hit and time.time() - hit[0] < CACHE_SECONDS:
        return hit[1], hit[2]

    t = yf.Ticker(symbol)
    hist = t.history(period="5d", interval="1m", auto_adjust=True)
    if hist.empty:
        raise LookupError(f"No minute data for {symbol}")
    closes = hist["Close"].dropna()
    closes = closes[closes.index.second == 0]                          # drop Yahoo's partial "live" row
    now = pd.Timestamp.now(tz=closes.index.tz)
    closes = closes[closes.index + pd.Timedelta(minutes=1) <= now]     # drop the bar still forming
    meta = t.history_metadata or {}
    with _lock:
        _bars[symbol] = (time.time(), closes, meta)
    return closes, meta


def available_models() -> dict:
    return {k: m for k, m in MODELS.items() if m.available}


def get_replay(symbol: str, closes: pd.Series) -> pd.DataFrame:
    """Every model's forecast for each of the last REPLAY_BARS bars, using only earlier bars."""
    last = closes.index[-1]
    with _lock:
        hit = _replays.get(symbol)
    if hit and hit[0] == last:
        return hit[1]
    t = closes.index
    idxs = [i for i in range(max(600, len(closes) - REPLAY_BARS - 1), len(closes) - 1)
            if t[i + 1] - t[i] <= MAX_GAP]
    frames = []
    if idxs:
        for key, m in available_models().items():
            with _model_lock:
                q = m.replay(closes, idxs)
            df = pd.DataFrame(q, columns=QCOLS)
            df["model"], df["time"] = key, t[[i + 1 for i in idxs]]
            df["last"], df["actual"] = closes.to_numpy()[idxs], closes.to_numpy()[[i + 1 for i in idxs]]
            frames.append(df.dropna(subset=QCOLS))
    rp = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    with _lock:
        _replays[symbol] = (last, rp)
    return rp


def update_live(symbol: str, closes: pd.Series) -> tuple[dict, list[dict]]:
    """Lock in every model's forecast the first time a bar is seen; score it when the next bar lands."""
    base = closes.index[-1]
    with _lock:
        have = base in _live.get(symbol, {})
    if not have:
        q = {}
        for key, m in available_models().items():
            with _model_lock:
                row = m.quantiles([closes])[0]
            if np.all(np.isfinite(row)):
                q[key] = row
        with _lock:
            _live.setdefault(symbol, {})[base] = {
                "made_at": datetime.now(timezone.utc), "last": float(closes.iloc[-1]), "q": q}

    results = []
    with _lock:
        preds = dict(_live[symbol])
    for b, p in sorted(preds.items()):
        later = closes[closes.index > b]
        if later.empty:
            continue
        nxt = later.index[0]
        results.append({"base": b, "time": nxt, "last": p["last"], "actual": float(later.iloc[0]),
                        "gap": nxt - b > MAX_GAP, "q": p["q"]})
    return preds[base], results


def feed_latency(symbol: str, closes: pd.Series) -> float:
    """Seconds between a bar closing and Yahoo publishing it, learned as new bars appear."""
    newest = closes.index[-1]
    since_close = (pd.Timestamp.now(tz=newest.tz) - newest).total_seconds() - 60
    with _lock:
        feed = _feed.get(symbol)
        if feed is None:
            # First look: a fresh feed is assumed ~20s behind; a stale one is at least this far behind.
            feed = _feed[symbol] = {"bar": newest, "latency": since_close if since_close > 120 else 20.0}
        elif newest > feed["bar"]:
            if newest - feed["bar"] <= pd.Timedelta(minutes=2):      # we watched it arrive
                feed["latency"] = 0.5 * feed["latency"] + 0.5 * max(since_close, 0.0)
            feed["bar"] = newest
        return feed["latency"]


def _hhmm(ts: pd.Timestamp) -> str:
    return ts.strftime("%H:%M")


def _clean(x):
    if isinstance(x, dict):
        return {k: _clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_clean(v) for v in x]
    if isinstance(x, (float, np.floating)):
        return float(x) if np.isfinite(x) else None
    return x


def _qdict(q) -> dict:
    return {c: float(v) for c, v in zip(QCOLS, q)}


def _inside(q, actual) -> bool:
    return bool(q[0] <= actual <= q[-1])


def _width(qs: dict) -> float:
    """The round's shared outer width: the average of the models' own 10th-90th percentile widths."""
    return float(np.mean([q[-1] - q[0] for q in qs.values()]))


def _game(q, width: float, actual: float | None = None) -> dict:
    """One model's game ranges for a round (and its points, once the price is known)."""
    olo, ohi, ilo, ihi = game_ranges(q[4], width)
    out = {"center": float(q[4]), "outer": [float(olo), float(ohi)], "inner": [float(ilo), float(ihi)],
           "own": [float(q[0]), float(q[-1])]}
    if actual is not None:
        out["points"] = int(points(q[4], width, actual))
    return out


@app.get("/")
def index():
    return send_from_directory(app.static_folder, "index.html")


@app.get("/api/markets")
def markets():
    return jsonify({"markets": catalog(), "utc": datetime.now(timezone.utc).isoformat()})


@app.get("/api/forecast/<symbol>")
def forecast_api(symbol: str):
    symbol = symbol.upper().strip()
    try:
        closes, meta = get_bars(symbol)
    except Exception as e:
        return jsonify({"error": f"Couldn't load {symbol}: {e}"}), 404
    if len(closes) < 700:
        return jsonify({"error": f"Not enough minute history for {symbol} yet (need about 2 sessions)"}), 404

    replay = get_replay(symbol, closes)
    rp = add_game_points(replay, ["time"]) if not replay.empty else replay
    pred, results = update_live(symbol, closes)
    keys = list(available_models())

    m = market_for(symbol)
    latency = feed_latency(symbol, closes)
    target = closes.index[-1] + pd.Timedelta(minutes=1)
    expected_in = (target + pd.Timedelta(minutes=1, seconds=latency)
                   - pd.Timestamp.now(tz=target.tz)).total_seconds()
    age = (pd.Timestamp.now(tz=closes.index.tz) - closes.index[-1]).total_seconds() - 60

    # chart: last hour of actual prices with each model's replayed 8-in-10 range
    by_model = {k: replay[replay["model"] == k].set_index("time") for k in keys} if not replay.empty else {}
    chart_points = []
    for t, c in closes.iloc[-60:].items():
        p = {"t": _hhmm(t), "actual": float(c), "ranges": {}}
        for k, df in by_model.items():
            if t in df.index:
                p["ranges"][k] = [float(df.at[t, "q10"]), float(df.at[t, "q90"])]
        chart_points.append(p)

    # live: every scored forecast this session, per model
    scored = [r for r in results if not r["gap"]]
    live_scores = {}
    for k in keys:
        rows = pd.DataFrame([{**_qdict(r["q"][k]), "last": r["last"], "actual": r["actual"],
                              "points": int(points(r["q"][k][4], _width(r["q"]), r["actual"]))}
                             for r in scored if k in r["q"]])
        live_scores[k] = score(rows)
    last_result = None
    if scored:
        r = scored[-1]
        last_result = {"target": _hhmm(r["time"]), "last": r["last"], "actual": r["actual"], "round": len(scored),
                       "width": _width(r["q"]),
                       "models": {k: _game(r["q"][k], _width(r["q"]), r["actual"]) for k in keys if k in r["q"]}}
    streak = {k: [{"t": _hhmm(r["time"]), "points": int(points(r["q"][k][4], _width(r["q"]), r["actual"]))}
                  for r in scored[-30:] if k in r["q"]] for k in keys}
    log = [{"target": _hhmm(r["time"]), "actual": r["actual"], "gap": bool(r["gap"]),
            "models": {k: _game(r["q"][k], _width(r["q"]), r["actual"]) for k in keys if k in r["q"]}}
           for r in reversed(results[-30:])]

    day_open = closes[closes.index.normalize() == closes.index[-1].normalize()].iloc[0]
    return jsonify(_clean({
        "symbol": symbol,
        "name": meta.get("longName") or meta.get("shortName") or symbol,
        "exchange": m["exchange"] if m else meta.get("exchangeName", ""),
        "currency": meta.get("currency", ""),
        "tz": str(closes.index.tz),
        "market": status(m) if m else None,
        "last_price": float(closes.iloc[-1]),
        "last_bar": _hhmm(closes.index[-1]),
        "session_change": float(closes.iloc[-1] / day_open - 1),
        "data_age_sec": max(0.0, age),
        "models": [MODELS[k].info() for k in MODELS],
        "report": REPORT,
        "forecast": {"target": _hhmm(target), "last": pred["last"],
                     "width": _width(pred["q"]) if pred["q"] else None,
                     "models": {k: {**_qdict(q), **_game(q, _width(pred["q"]))} for k, q in pred["q"].items()}},
        "points": chart_points,
        "replay": ({k: score(rp[rp["model"] == k]) for k in keys} if not replay.empty else {k: {"n": 0} for k in keys}),
        "live": {
            "round": len(scored) + 1,
            "made_at_local": pred["made_at"].astimezone(closes.index.tz).strftime("%H:%M:%S"),
            "made_at": pred["made_at"].isoformat(),
            "expected_in_sec": expected_in,
            "latency_sec": latency,
            "last_result": last_result,
            "streak": streak,
            "scores": live_scores,
            "log": log,
        },
    }))


def serve(models: dict, report: dict | None = None, port: int = 5050, host: str = "127.0.0.1",
          background: bool = False):
    """Run the app with the given forecasters (the Colab notebook runs it in the background)."""
    global MODELS, REPORT
    MODELS, REPORT = models, report
    run = lambda: app.run(host=host, port=port, threaded=True, use_reloader=False)
    if background:
        th = threading.Thread(target=run, daemon=True)
        th.start()
        return th
    run()


if __name__ == "__main__":
    import argparse

    from riskmc.forecasters import build

    p = argparse.ArgumentParser()
    p.add_argument("--no-ai", action="store_true", help="Monte Carlo only")
    p.add_argument("--models-dir", default=str(ROOT / "models"))
    a = p.parse_args()
    report_path = Path(a.models_dir) / "report.json"
    report = json.loads(report_path.read_text()) if report_path.exists() else None
    chronos_dir, timesfm_dir = Path(a.models_dir) / "chronos2-lora", Path(a.models_dir) / "timesfm-lora"
    models = build(load_ai=not a.no_ai,
                   chronos_path=str(chronos_dir) if chronos_dir.exists() else None,
                   timesfm_adapter=str(timesfm_dir) if timesfm_dir.exists() else None,
                   scales=(report or {}).get("scales"), configs=(report or {}).get("configs"))
    for m in models.values():
        info = m.info()
        state = "not loaded" if not m.available else "trained" if info["fine_tuned"] else "base model"
        setup = f", {info['input_mode']} input, context {info['context']}" if "input_mode" in info else ""
        print(f"  {m.name:<12} {state}{setup}, range width x{m.scale:.2f}")
    print("Forecast Arena → http://127.0.0.1:5050")
    serve(models, report)

In [ ]:
%%writefile /content/lab/web/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Forecast Arena</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Chakra+Petch:wght@500;600;700&family=IBM+Plex+Mono:wght@400;500&family=IBM+Plex+Sans:wght@400;500;600&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
/* Layout: a match screen. Round header → leaderboard → aim / result boards → round history → pro stats (collapsed). */
:root {
  --bg: #eef1f4; --panel: #ffffff; --ink: #121820; --muted: #566170; --faint: #8994a3; --line: #dde2e8; --track: #e9edf1;
  --s-mc: #2a78d6; --s-chronos2: #eb6834; --s-timesfm: #1baf7a;
  --gold: #c98a00; --gold-soft: #fbefcf; --hit: #1a7f4b; --hit-soft: #dcf1e5; --miss: #8994a3; --miss-soft: #eef1f4; --bad: #c0392b;
  --silver: #8d96a1; --bronze: #b06a35;
  --display: "Chakra Petch", "IBM Plex Sans", system-ui, sans-serif;
  --sans: "IBM Plex Sans", system-ui, -apple-system, "Segoe UI", sans-serif;
  --mono: "IBM Plex Mono", ui-monospace, "SF Mono", Menlo, monospace;
}
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    --bg: #0b0f14; --panel: #131a22; --ink: #e8edf3; --muted: #9aa6b4; --faint: #687584; --line: #222c37; --track: #1a222c;
    --s-mc: #3987e5; --s-chronos2: #d95926; --s-timesfm: #199e70;
    --gold: #f2b630; --gold-soft: #3a2c0c; --hit: #46c083; --hit-soft: #13301f; --miss: #687584; --miss-soft: #1a222c; --bad: #f07167;
    --silver: #aab3bd; --bronze: #cf8a52;
    color-scheme: dark;
  }
}
:root[data-theme="dark"] {
  --bg: #0b0f14; --panel: #131a22; --ink: #e8edf3; --muted: #9aa6b4; --faint: #687584; --line: #222c37; --track: #1a222c;
  --s-mc: #3987e5; --s-chronos2: #d95926; --s-timesfm: #199e70;
  --gold: #f2b630; --gold-soft: #3a2c0c; --hit: #46c083; --hit-soft: #13301f; --miss: #687584; --miss-soft: #1a222c; --bad: #f07167;
  --silver: #aab3bd; --bronze: #cf8a52;
  color-scheme: dark;
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--ink); font: 15px/1.5 var(--sans); }
.num { font-family: var(--mono); font-variant-numeric: tabular-nums; }
.disp { font-family: var(--display); letter-spacing: .02em; }
.wrap { max-width: 1040px; margin: 0 auto; padding-block: 18px 48px;
  padding-left: max(16px, env(safe-area-inset-left, 0px)); padding-right: max(16px, env(safe-area-inset-right, 0px)); }

.top { display: flex; flex-wrap: wrap; align-items: center; gap: 10px 16px; margin-bottom: 18px; }
.brand { font-family: var(--display); font-weight: 700; font-size: 20px; letter-spacing: .12em; margin-right: auto; text-transform: uppercase; }
.brand span { color: var(--gold); }
select, input, button { font: inherit; color: var(--ink); background: var(--panel); border: 1px solid var(--line); border-radius: 8px; padding: 8px 12px; }
select { min-width: 0; max-width: 100%; cursor: pointer; }
input { width: 160px; }
button { cursor: pointer; }
:focus-visible { outline: 2px solid var(--gold); outline-offset: 2px; }
.picker { display: flex; flex-wrap: wrap; gap: 8px; min-width: 0; }
.custom { display: flex; gap: 6px; }

.card { background: var(--panel); border: 1px solid var(--line); border-radius: 16px; padding: 20px 22px; min-width: 0; }
.stack { display: flex; flex-direction: column; gap: 14px; }
.eyebrow { font-family: var(--display); font-size: 12px; letter-spacing: .14em; text-transform: uppercase; color: var(--faint); font-weight: 600; }
.sub { color: var(--muted); font-size: 13px; }
h2 { margin: 0; font-family: var(--display); font-size: 17px; font-weight: 600; letter-spacing: .03em; }
.card-head { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; gap: 8px 16px; margin-bottom: 14px; }
.chip { display: inline-flex; align-items: center; gap: 5px; font-size: 12px; font-weight: 500; padding: 2px 9px; border-radius: 99px; white-space: nowrap; }
.chip.open { color: var(--hit); background: var(--hit-soft); }
.chip.delayed { color: var(--gold); background: var(--gold-soft); }
.chip.closed { color: var(--muted); background: var(--track); }

/* Match header */
.match { display: grid; grid-template-columns: 1fr auto; gap: 16px 28px; align-items: center; }
.match h1 { margin: 0; font-family: var(--display); font-size: 22px; font-weight: 600; text-wrap: balance; }
.price-row { display: flex; align-items: baseline; gap: 12px; margin-top: 6px; flex-wrap: wrap; }
.price { font-size: 26px; font-weight: 500; }
.round { display: flex; align-items: center; gap: 16px; }
.round-label { text-align: right; }
.round-no { font-family: var(--display); font-weight: 700; font-size: 34px; line-height: 1; }
.ring { position: relative; width: 74px; height: 74px; flex: none; }
.ring svg { transform: rotate(-90deg); }
.ring .bg { stroke: var(--track); }
.ring .fg { stroke: var(--gold); transition: stroke-dashoffset 1s linear; }
.ring .t { position: absolute; inset: 0; display: grid; place-items: center; font-family: var(--display); font-weight: 600; font-size: 17px; }

/* Leaderboard */
.players { display: grid; grid-template-columns: repeat(3, 1fr); gap: 14px; }
.player { position: relative; background: var(--panel); border: 1px solid var(--line); border-radius: 16px; padding: 16px 18px 14px; overflow: hidden; }
.player::before { content: ""; position: absolute; inset: 0 0 auto; height: 4px; background: var(--pc); }
.player.lead { border-color: var(--gold); box-shadow: 0 0 0 1px var(--gold) inset; }
.p-top { display: flex; align-items: center; gap: 10px; }
.avatar { width: 38px; height: 38px; border-radius: 10px; display: grid; place-items: center; font-family: var(--display); font-weight: 700; font-size: 14px;
  color: #fff; background: var(--pc); flex: none; }
.p-name { font-weight: 600; line-height: 1.2; }
.p-sub { font-size: 12px; color: var(--muted); }
.rank { margin-left: auto; font-family: var(--display); font-weight: 700; font-size: 13px; padding: 3px 9px; border-radius: 8px; letter-spacing: .06em; }
.rank.r1 { color: var(--gold); background: var(--gold-soft); }
.rank.r2 { color: var(--silver); background: var(--track); }
.rank.r3 { color: var(--bronze); background: var(--track); }
.p-score { display: flex; align-items: baseline; gap: 8px; margin-top: 12px; }
.p-pts { font-family: var(--display); font-weight: 700; font-size: 44px; line-height: 1; }
.p-unit { color: var(--muted); font-size: 14px; }
.p-last { margin-left: auto; }
.badge { display: inline-flex; align-items: center; gap: 4px; font-family: var(--display); font-weight: 600; font-size: 12px; letter-spacing: .06em;
  padding: 3px 9px; border-radius: 8px; text-transform: uppercase; white-space: nowrap; }
.badge.b2 { color: var(--gold); background: var(--gold-soft); }
.badge.b1 { color: var(--hit); background: var(--hit-soft); }
.badge.b0 { color: var(--miss); background: var(--miss-soft); }
.p-stats { display: flex; gap: 14px; margin-top: 12px; padding-top: 10px; border-top: 1px solid var(--line); font-size: 12px; color: var(--muted); flex-wrap: wrap; }
.p-stats b { color: var(--ink); font-weight: 600; font-family: var(--mono); }
.pop { position: absolute; right: 18px; top: 52px; font-family: var(--display); font-weight: 700; font-size: 26px; color: var(--gold);
  animation: pop 1.6s ease-out forwards; pointer-events: none; }
.pop.p1 { color: var(--hit); } .pop.p0 { color: var(--miss); }
@keyframes pop { 0% { opacity: 0; transform: translateY(10px) scale(.8); } 15% { opacity: 1; transform: translateY(0) scale(1.15); }
  70% { opacity: 1; transform: translateY(-14px) scale(1); } 100% { opacity: 0; transform: translateY(-24px); } }
.warmup { font-size: 12px; color: var(--faint); margin-top: 6px; }

/* Boards */
.boards { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
.lanes { display: grid; grid-template-columns: 40px 1fr; gap: 14px 12px; align-items: center; }
.lane { position: relative; height: 30px; }
.lane .track { position: absolute; inset: 14px 0 auto; height: 2px; background: var(--line); }
.lane .outer { position: absolute; top: 4px; height: 22px; border-radius: 6px; background: var(--pc); opacity: .28; }
.lane .inner { position: absolute; top: 4px; height: 22px; border-radius: 4px; background: var(--pc); }
.lane .vline { position: absolute; top: -8px; bottom: -8px; width: 0; border-left: 2px dashed var(--faint); }
.lane .actual { position: absolute; top: -10px; bottom: -10px; width: 3px; margin-left: -1.5px; background: var(--ink); border-radius: 2px; }
.mini-avatar { width: 32px; height: 32px; border-radius: 8px; display: grid; place-items: center; font-family: var(--display); font-weight: 700;
  font-size: 12px; color: #fff; background: var(--pc); position: relative; }
.mini-avatar .res { position: absolute; right: -10px; top: -8px; }
.axis { grid-column: 2; display: flex; justify-content: space-between; font-size: 11px; color: var(--faint); margin-top: -4px; }
.board-note { font-size: 12px; color: var(--muted); margin-top: 12px; }
.result-board.reveal .actual { animation: drop .7s cubic-bezier(.2, 1.4, .4, 1); }
@keyframes drop { from { transform: translateY(-26px); opacity: 0; } to { transform: none; opacity: 1; } }
.res-pill { font-family: var(--display); font-weight: 700; font-size: 11px; padding: 1px 6px; border-radius: 6px; }
.res-pill.r2 { color: var(--gold); background: var(--gold-soft); } .res-pill.r1 { color: var(--hit); background: var(--hit-soft); } .res-pill.r0 { color: var(--miss); background: var(--miss-soft); }

/* Round history */
.history { display: grid; grid-template-columns: max-content 1fr; gap: 8px 12px; align-items: center; }
.cells { display: flex; gap: 3px; flex-wrap: wrap; }
.cell { width: 20px; height: 20px; border-radius: 5px; display: grid; place-items: center; font-family: var(--display); font-weight: 700; font-size: 11px; }
.cell.c2 { background: var(--gold); color: var(--panel); }
.cell.c1 { background: var(--hit-soft); color: var(--hit); }
.cell.c0 { background: var(--miss-soft); color: var(--miss); }
.cell.empty { border: 1px dashed var(--line); }
.rules { display: flex; flex-wrap: wrap; gap: 8px 18px; font-size: 13px; color: var(--muted); }
.rules b { color: var(--ink); }

/* Pro stats */
details { border-top: 1px solid var(--line); padding: 12px 0; }
details:first-of-type { border-top: 0; }
summary { cursor: pointer; font-weight: 500; }
details p { color: var(--muted); max-width: 72ch; margin: 10px 0; }
.seg { display: inline-flex; background: var(--track); border-radius: 9px; padding: 3px; }
.seg button { border: 0; background: transparent; padding: 5px 12px; border-radius: 7px; font-size: 13px; color: var(--muted); }
.seg button[aria-pressed="true"] { background: var(--panel); color: var(--ink); font-weight: 500; }
.table-wrap { overflow-x: auto; margin-top: 10px; }
table { width: 100%; border-collapse: collapse; font-size: 13px; }
th, td { padding: 8px 10px; text-align: right; border-bottom: 1px solid var(--line); white-space: nowrap; }
th { color: var(--faint); font-weight: 600; font-size: 12px; }
th small { display: block; font-weight: 400; }
th:first-child, td:first-child { text-align: left; }
.chart-box { position: relative; height: 280px; margin-top: 10px; }
.sw { display: inline-block; width: 10px; height: 10px; border-radius: 3px; margin-right: 6px; vertical-align: middle; }

.state { text-align: center; padding: 72px 16px; color: var(--muted); }
.state h2 { color: var(--ink); font-size: 22px; margin: 0 0 6px; }
.err { color: var(--bad); }
.updated { color: var(--faint); font-size: 12px; }

@media (max-width: 860px) {
  .players { grid-template-columns: 1fr; }
  .boards { grid-template-columns: 1fr; }
}
@media (max-width: 600px) {
  .card { padding: 16px; }
  .match { grid-template-columns: 1fr; }
  .round { justify-content: space-between; }
  .round-label { text-align: left; }
  .picker, .picker select { width: 100%; }
  .custom { width: 100%; } .custom input { flex: 1; }
  .p-pts { font-size: 38px; }
}
@media (prefers-reduced-motion: reduce) { * { transition: none !important; animation: none !important; } }
</style>
</head>
<body>
<div class="wrap">
  <div class="top">
    <div class="brand">Forecast <span>Arena</span></div>
    <div class="picker">
      <select id="ticker" aria-label="Choose a ticker"><option>Loading markets…</option></select>
      <form class="custom" id="custom">
        <input id="symbol" placeholder="Other symbol" aria-label="Any Yahoo Finance symbol" autocomplete="off">
        <button type="submit">Play</button>
      </form>
    </div>
  </div>
  <main id="main" class="stack">
    <div class="card state"><h2 class="disp">Pick a market to start a match</h2>Markets that are open right now are listed first.</div>
  </main>
</div>

<script>
const POLL_MS = 10000;
const INITIALS = { mc: 'MC', chronos2: 'C2', timesfm: 'TF' };
let current = null, pollTimer = null, chart = null, lastUpdate = 0, liveClock = null;
let lastRound = null, revealUntil = 0, pops = {};
let statsTab = 'replay';
try { statsTab = localStorage.getItem('fa-stats') || 'replay'; } catch {}

const tok = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const esc = s => String(s ?? '').replace(/[&<>"]/g, c => ({'&': '&amp;', '<': '&lt;', '>': '&gt;', '"': '&quot;'}[c]));
const pct = x => x == null ? '–' : Math.round(x * 100) + '%';
let digits = 2;
const fmt = x => x == null ? '–' : x.toLocaleString(undefined, { minimumFractionDigits: digits, maximumFractionDigits: digits });
const signed = (x, d = 2) => (x > 0 ? '+' : x < 0 ? '−' : '') + Math.abs(x).toFixed(d);
const pc = k => `--pc:var(--s-${esc(k)})`;
const RESULT = { 2: ['b2', '🎯 Bullseye', '+2'], 1: ['b1', 'Hit', '+1'], 0: ['b0', 'Miss', '0'] };

/* ---------- picker ---------- */
async function loadMarkets() {
  const { markets } = await fetch('api/markets').then(r => r.json());
  const label = { open: 'open now', break: 'lunch break', closed: 'closed' };
  const sel = document.getElementById('ticker');
  sel.innerHTML = '<option value="">Choose a market…</option>' + markets.map(m =>
    `<optgroup label="${esc(m.name)} · ${esc(m.exchange)} · ${label[m.state]}">${m.tickers.map(t =>
      `<option value="${esc(t.symbol)}">${esc(t.name)} (${esc(t.symbol)})</option>`).join('')}</optgroup>`).join('');
  if (current) {
    if (![...sel.options].some(o => o.value === current)) sel.add(new Option(current, current));
    sel.value = current;
  }
  return markets;
}
document.getElementById('ticker').addEventListener('change', e => e.target.value && select(e.target.value));
document.getElementById('custom').addEventListener('submit', e => {
  e.preventDefault();
  const v = document.getElementById('symbol').value.trim().toUpperCase();
  if (v) { select(v); document.getElementById('symbol').value = ''; }
});

function select(sym) {
  current = sym; lastUpdate = 0; lastRound = null; pops = {};
  if (chart) { chart.destroy(); chart = null; }
  try { localStorage.setItem('fa-ticker', sym); } catch {}
  const sel = document.getElementById('ticker');
  if (![...sel.options].some(o => o.value === sym)) sel.add(new Option(sym, sym));
  sel.value = sym;
  document.getElementById('main').innerHTML = `<div class="card state"><h2 class="disp">Loading ${esc(sym)}</h2>Fetching minute prices and playing warm-up rounds over the last two hours…</div>`;
  clearTimeout(pollTimer); clearInterval(liveClock); poll();
}
function schedulePoll(ms) { clearTimeout(pollTimer); pollTimer = setTimeout(poll, ms); }

async function poll() {
  const sym = current;
  let d;
  try { d = await fetch('api/forecast/' + encodeURIComponent(sym)).then(r => r.json()); }
  catch { d = { error: 'Lost connection to the server. Retrying every 10 seconds.' }; }
  if (sym !== current) return;
  schedulePoll(POLL_MS);
  if (d.error) {
    if (!lastUpdate) document.getElementById('main').innerHTML =
      `<div class="card state"><h2 class="disp">Can't start a match on ${esc(sym)}</h2><span class="err">${esc(d.error)}</span><br>Check the symbol on finance.yahoo.com, or pick one from the list.</div>`;
    return;
  }
  lastUpdate = Date.now();
  render(d);
}

/* ---------- helpers ---------- */
const marketState = d => d.market ? d.market.state : (d.data_age_sec < 600 ? 'open' : 'closed');
function statusChip(d) {
  const st = marketState(d);
  if (st === 'break') return `<span class="chip closed">Lunch break</span>`;
  if (st !== 'open') return `<span class="chip closed">Market closed</span>`;
  if (d.data_age_sec > 120) return `<span class="chip delayed">Live · prices ${Math.round(d.data_age_sec / 60)} min delayed</span>`;
  return `<span class="chip open">Live</span>`;
}
function streakOf(list) { let n = 0; for (let i = list.length - 1; i >= 0 && list[i].points > 0; i--) n++; return n; }
function counts(list) { const c = [0, 0, 0]; list.forEach(x => c[x.points]++); return c; }

function standings(d) {
  const ms = d.models.filter(m => m.available);
  const rows = ms.map(m => {
    const s = d.live.streak[m.key] || [], c = counts(s), live = d.live.scores[m.key] || {}, warm = d.replay[m.key] || {};
    return { m, s, c, pts: live.points || 0, rounds: live.n || 0, warm, streak: streakOf(s) };
  });
  const anyLive = rows.some(r => r.rounds);
  rows.sort((a, b) => anyLive ? b.pts - a.pts || (b.warm.points_per_round || 0) - (a.warm.points_per_round || 0)
                              : (b.warm.points_per_round || 0) - (a.warm.points_per_round || 0));
  return { rows, anyLive };
}

/* ---------- sections ---------- */
function matchHeader(d) {
  const L = d.live, open = marketState(d) === 'open';
  return `<section class="card match">
    <div>
      <div class="eyebrow">${esc(d.exchange)} · <span class="num">${esc(d.symbol)}</span></div>
      <h1>${esc(d.name)}</h1>
      <div class="price-row"><span class="price num">${fmt(d.last_price)}${d.currency ? ' <span class="sub">' + esc(d.currency) + '</span>' : ''}</span>
        <span class="sub"><span style="color:${d.session_change >= 0 ? 'var(--hit)' : 'var(--bad)'}">${signed(d.session_change * 100)}%</span> today</span>
        ${statusChip(d)}</div>
    </div>
    <div class="round">
      <div class="round-label"><div class="eyebrow">Round</div><div class="round-no">${L.round}</div>
        <div class="sub">${open ? `Price at <b class="num" style="color:var(--ink)">${esc(d.forecast.target)}</b>` : 'Paused until the market opens'}</div></div>
      <div class="ring" aria-label="Time until the next price">
        <svg width="74" height="74" viewBox="0 0 74 74"><circle class="bg" cx="37" cy="37" r="31" fill="none" stroke-width="7"/>
          <circle class="fg" id="ring-fg" cx="37" cy="37" r="31" fill="none" stroke-width="7" stroke-linecap="round"
            stroke-dasharray="194.8" stroke-dashoffset="${open ? 194.8 : 0}"/></svg>
        <div class="t" id="ring-t">${open ? '…' : '⏸'}</div>
      </div>
    </div>
  </section>`;
}

function leaderboard(d) {
  const { rows, anyLive } = standings(d);
  const medal = ['1ST', '2ND', '3RD'];
  return `<section class="players" aria-label="Leaderboard">${rows.map((r, i) => {
    const last = r.s.length ? r.s[r.s.length - 1].points : null;
    const pop = pops[r.m.key];
    return `<div class="player ${i === 0 && (anyLive ? r.pts > 0 : true) ? 'lead' : ''}" style="${pc(r.m.key)}">
      <div class="p-top"><div class="avatar">${INITIALS[r.m.key] || '?'}</div>
        <div><div class="p-name">${esc(r.m.name)}</div><div class="p-sub">${r.m.fine_tuned ? 'Fine-tuned' : r.m.key === 'mc' ? 'Simulation' : 'Base model'}${r.m.input_mode ? ' · ' + esc(r.m.input_mode) : ''}</div></div>
        <span class="rank r${i + 1}">${medal[i]}</span></div>
      <div class="p-score"><span class="p-pts" data-count="${r.pts}">${r.pts}</span><span class="p-unit">pts</span>
        <span class="p-last">${last == null ? '' : `<span class="badge b${last}">${RESULT[last][1]}</span>`}</span></div>
      <div class="p-stats"><span><b>${r.rounds}</b> rounds</span><span>🎯 <b>${r.c[2]}</b></span><span>Hits <b>${r.c[1]}</b></span>
        <span>Misses <b>${r.c[0]}</b></span>${r.streak >= 3 ? `<span>🔥 <b>${r.streak}</b> streak</span>` : ''}</div>
      <div class="warmup">Warm-up, last 2 h: <span class="num">${r.warm.n ? r.warm.points_per_round.toFixed(2) : '–'}</span> pts/round · 🎯 ${pct(r.warm.bullseye)}</div>
      ${pop != null ? `<div class="pop p${pop}">${RESULT[pop][2]}</div>` : ''}
    </div>`;
  }).join('')}</section>`;
}

function lanes(entries, a, b, opts = {}) {
  const x = v => ((v - a) / (b - a) * 100).toFixed(2) + '%';
  const w = (p, q) => ((q - p) / (b - a) * 100).toFixed(2) + '%';
  return entries.map(({ m, g }) => `
    <div class="mini-avatar" style="${pc(m.key)}" title="${esc(m.name)}">${INITIALS[m.key] || '?'}
      ${opts.actual != null && g.points != null ? `<span class="res res-pill r${g.points}">${RESULT[g.points][2]}</span>` : ''}</div>
    <div class="lane" style="${pc(m.key)}" title="${esc(m.name)}: aim ${fmt(g.center)} · bullseye ${fmt(g.inner[0])}–${fmt(g.inner[1])} · range ${fmt(g.outer[0])}–${fmt(g.outer[1])}">
      <div class="track"></div>
      <div class="outer" style="left:${x(g.outer[0])};width:${w(g.outer[0], g.outer[1])}"></div>
      <div class="inner" style="left:${x(g.inner[0])};width:${w(g.inner[0], g.inner[1])}"></div>
      ${opts.now != null ? `<div class="vline" style="left:${x(opts.now)}"></div>` : ''}
      ${opts.actual != null ? `<div class="actual" style="left:${x(opts.actual)}"></div>` : ''}
    </div>`).join('');
}

function bounds(vals) {
  const lo = Math.min(...vals), hi = Math.max(...vals), pad = (hi - lo) * 0.1 || Math.abs(lo) * 1e-4;
  return [lo - pad, hi + pad];
}

function boards(d) {
  const ms = d.models.filter(m => m.available);
  const f = d.forecast, r = d.live.last_result;
  const aimEntries = ms.filter(m => f.models[m.key]).map(m => ({ m, g: f.models[m.key] }));
  const [a, b] = bounds([f.last, ...aimEntries.flatMap(e => e.g.outer)]);
  const aim = `<section class="card">
    <div class="card-head"><h2>Round ${d.live.round} · Aim</h2><span class="sub">Locked in at <span class="num">${esc(d.live.made_at_local)}</span></span></div>
    ${aimEntries.length ? `<div class="lanes">${lanes(aimEntries, a, b, { now: f.last })}
      <div class="axis num"><span>${fmt(a)}</span><span>┆ now ${fmt(f.last)}</span><span>${fmt(b)}</span></div></div>` : '<p class="sub">Waiting for the models…</p>'}
    <div class="board-note">Solid block = 🎯 bullseye zone (2 pts) · faded band = range (1 pt). Every player gets the same width this round${f.width ? ` (<span class="num">${fmt(f.width)}</span>)` : ''}; only the aim differs.</div>
  </section>`;
  let res;
  if (r) {
    const entries = ms.filter(m => r.models[m.key]).map(m => ({ m, g: r.models[m.key] }));
    const [ra, rb] = bounds([r.actual, ...entries.flatMap(e => e.g.outer)]);
    res = `<section class="card result-board ${Date.now() < revealUntil ? 'reveal' : ''}">
      <div class="card-head"><h2>Round ${r.round} · Result</h2><span class="sub">Price at ${esc(r.target)}: <b class="num" style="color:var(--ink)">${fmt(r.actual)}</b></span></div>
      <div class="lanes">${lanes(entries, ra, rb, { actual: r.actual })}
        <div class="axis num"><span>${fmt(ra)}</span><span>▮ actual price</span><span>${fmt(rb)}</span></div></div>
    </section>`;
  } else {
    res = `<section class="card"><div class="card-head"><h2>Result</h2></div>
      <p class="sub">The first live round is scored when the ${esc(f.target)} price arrives. Until then, the warm-up score on each card shows how every player did over the last two hours.</p></section>`;
  }
  return `<div class="boards">${aim}${res}</div>`;
}

function history(d) {
  const ms = d.models.filter(m => m.available);
  const rows = ms.map(m => {
    const s = (d.live.streak[m.key] || []).slice(-24);
    const cells = s.map(x => `<span class="cell c${x.points}" title="${esc(x.t)}: ${RESULT[x.points][1]}">${x.points === 0 ? '·' : x.points}</span>`).join('')
      + Array.from({ length: Math.max(0, 12 - s.length) }, () => '<span class="cell empty"></span>').join('');
    return `<div class="mini-avatar" style="${pc(m.key)}" title="${esc(m.name)}">${INITIALS[m.key] || '?'}</div><div class="cells">${cells}</div>`;
  }).join('');
  return `<section class="card">
    <div class="card-head"><h2>Round history</h2><span class="updated" id="updated"></span></div>
    <div class="history">${rows}</div>
    <div class="rules" style="margin-top:14px"><span><span class="cell c2" style="display:inline-grid">2</span> <b>Bullseye</b>, inside the inner zone (20% of the range)</span>
      <span><span class="cell c1" style="display:inline-grid">1</span> <b>Hit</b>, inside the range</span>
      <span><span class="cell c0" style="display:inline-grid">·</span> <b>Miss</b></span></div>
  </section>`;
}

function proStats(d) {
  const src = statsTab === 'live' ? d.live.scores : d.replay;
  const ms = d.models.filter(m => m.available && src[m.key] && src[m.key].n);
  const rep = d.report;
  const trainTable = rows => `<div class="table-wrap"><table><tr><th>Model</th><th>Pts / round</th><th>Bullseye</th><th>In own 80% range</th><th>Pinball (bps)</th></tr>
    ${rows.map(r => `<tr><td>${esc(r.model)}</td><td class="num">${r['points per round'] != null ? (+r['points per round']).toFixed(2) : '–'}</td>
      <td class="num">${pct(r['bullseye rate'])}</td><td class="num">${pct(r['in 8-in-10 range'])}</td><td class="num">${(+r['pinball loss (bps)']).toFixed(3)}</td></tr>`).join('')}</table></div>`;
  return `<section class="card" style="padding-block:6px">
    <details id="d-stats"><summary>Pro stats</summary>
      <div style="margin-top:10px" class="seg" role="group" aria-label="Stats source">
        <button data-tab="replay" aria-pressed="${statsTab === 'replay'}">Warm-up · last 2 hours</button>
        <button data-tab="live" aria-pressed="${statsTab === 'live'}">Live match</button></div>
      ${ms.length ? `<div class="table-wrap"><table>
        <tr><th>Model</th><th>Pts / round<small>game score</small></th><th>Bullseye<small>inner zone</small></th>
          <th>In own 80% range<small>target 80%</small></th><th>Own range width<small>bps</small></th><th>Pinball loss<small>bps, lower wins</small></th><th>Direction<small>coin flip 50%</small></th></tr>
        ${ms.map(m => { const s = src[m.key]; return `<tr><td><span class="sw" style="background:var(--s-${esc(m.key)})"></span>${esc(m.name)}</td>
          <td class="num">${s.points_per_round.toFixed(2)}</td><td class="num">${pct(s.bullseye)}</td><td class="num">${pct(s.coverage80)}</td>
          <td class="num">${s.width80_bps.toFixed(1)}</td><td class="num">${s.pinball_bps.toFixed(3)}</td><td class="num">${pct(s.direction_acc)}</td></tr>`; }).join('')}
      </table></div>` : '<p>No rounds scored yet.</p>'}
      <p>The game gives every player the same range width. These columns judge each model's own range as well: whether it catches about 80% of prices,
      and pinball loss, which rewards ranges that are both correct and narrow.</p>
      <div class="chart-box"><canvas id="chart" aria-label="Each model's own 80% range per minute with the actual price"></canvas></div>
    </details>
    <details id="d-train"><summary>Training results</summary>
      ${rep ? `<p>Trained ${esc(rep.trained_at)} on ${rep.tickers} tickers (${Number(rep.bars).toLocaleString()} one-minute prices); scored on the last 2 sessions, which no model trained on.</p>
        <p><b style="color:var(--ink)">Out of the box</b></p>${trainTable(rep.zero_shot)}
        ${rep.input_fixed ? `<p><b style="color:var(--ink)">Chosen input, untrained</b></p>${trainTable(rep.input_fixed)}` : ''}
        <p><b style="color:var(--ink)">Fine-tuned</b></p>${trainTable(rep.trained)}`
      : '<p>No training report loaded. Run the Colab notebook to fine-tune and score the models.</p>'}
    </details>
    <details id="d-how"><summary>About the players</summary>
      ${d.models.map(m => `<p><span class="sw" style="background:var(--s-${esc(m.key)})"></span><b style="color:var(--ink)">${esc(m.name)}</b>: ${esc(m.blurb || '')}${m.available ? '' : ' · not loaded'}</p>`).join('')}
      <p>Each round, every player predicts the next 1-minute price. Its aim is its median forecast. All players then get the same range width, the average of their own
      10th–90th percentile ranges, so the score only rewards aim: 2 points inside the bullseye zone, 1 inside the range.</p>
    </details>
  </section>`;
}

/* ---------- render ---------- */
function render(d) {
  digits = d.last_price < 10 ? 4 : 2;
  const r = d.live.last_result;
  if (r && lastRound != null && r.round !== lastRound) {               // a new round was just scored
    revealUntil = Date.now() + 1500;
    pops = Object.fromEntries(Object.entries(r.models).map(([k, g]) => [k, g.points]));
    setTimeout(() => { pops = {}; }, 1700);
  } else if (Date.now() > revealUntil) pops = {};
  if (r) lastRound = r.round; else if (lastRound == null) lastRound = 0;

  const open = ['d-stats', 'd-train', 'd-how'].filter(id => document.getElementById(id)?.open);
  if (chart) { chart.destroy(); chart = null; }
  document.getElementById('main').innerHTML = matchHeader(d) + leaderboard(d) + boards(d) + history(d) + proStats(d);
  open.forEach(id => document.getElementById(id).open = true);
  document.querySelectorAll('.seg button').forEach(b => b.addEventListener('click', () => {
    statsTab = b.dataset.tab; try { localStorage.setItem('fa-stats', statsTab); } catch {}
    render(d); document.getElementById('d-stats').open = true;
  }));
  document.getElementById('d-stats').addEventListener('toggle', e => { if (e.target.open && !chart) drawChart(d); });
  if (document.getElementById('d-stats').open) drawChart(d);
  startClock(d);
}

function drawChart(d) {
  const canvas = document.getElementById('chart');
  if (!canvas) return;
  const pts = d.points, ms = d.models.filter(m => m.available);
  const vals = pts.flatMap(p => [p.actual, ...ms.flatMap(m => p.ranges[m.key] || [])]).filter(v => v != null);
  const span = Math.max(...vals) - Math.min(...vals) || vals[0] * 1e-4;
  const datasets = ms.map(m => ({ type: 'bar', label: m.name, data: pts.map(p => p.ranges[m.key] || null),
    backgroundColor: tok('--s-' + m.key) + 'b3', borderRadius: 2, borderSkipped: false, barPercentage: .9, categoryPercentage: .85, order: 2 }));
  datasets.push({ type: 'line', label: 'Actual', data: pts.map(p => p.actual), showLine: false, order: 1, pointRadius: 3,
    pointBackgroundColor: tok('--ink'), pointBorderColor: tok('--panel'), pointBorderWidth: 1.5 });
  chart = new Chart(canvas, { data: { labels: pts.map(p => p.t), datasets }, options: {
    responsive: true, maintainAspectRatio: false, animation: false, interaction: { mode: 'index', intersect: false },
    plugins: { legend: { display: false }, tooltip: { filter: i => i.raw != null, callbacks: {
      label: c => c.dataset.type === 'bar' ? `${c.dataset.label}: ${fmt(c.raw[0])} – ${fmt(c.raw[1])}` : `Actual: ${fmt(c.raw)}` } } },
    scales: { x: { ticks: { color: tok('--faint'), maxTicksLimit: 8, maxRotation: 0 }, grid: { display: false } },
      y: { position: 'right', beginAtZero: false, suggestedMin: Math.min(...vals) - span * .06, suggestedMax: Math.max(...vals) + span * .06,
           ticks: { color: tok('--faint'), maxTicksLimit: 6, callback: v => fmt(v) }, grid: { color: tok('--line') } } } } });
}

function startClock(d) {
  clearInterval(liveClock);
  const L = d.live, open = marketState(d) === 'open';
  const startMs = new Date(L.made_at).getTime(), dueMs = Date.now() + L.expected_in_sec * 1000;
  const C = 194.8;
  let fast = false;
  const tick = () => {
    const up = document.getElementById('updated');
    if (up && lastUpdate) up.textContent = `Checked ${Math.round((Date.now() - lastUpdate) / 1000)}s ago`;
    if (!open) return;
    const left = Math.round((dueMs - Date.now()) / 1000);
    const t = document.getElementById('ring-t'), fg = document.getElementById('ring-fg');
    if (t) t.textContent = left > 0 ? (left >= 60 ? `${Math.floor(left / 60)}:${String(left % 60).padStart(2, '0')}` : `${left}s`) : '…';
    const frac = Math.min(1, Math.max(0, (Date.now() - startMs) / Math.max(1, dueMs - startMs)));
    if (fg) fg.setAttribute('stroke-dashoffset', (C * (1 - frac)).toFixed(1));
    if (left <= 0 && !fast) { fast = true; schedulePoll(3000); }       // price is due: check more often
  };
  tick(); liveClock = setInterval(tick, 1000);
}

(async () => {
  const markets = await loadMarkets();
  let start = decodeURIComponent(location.hash.slice(1)) || null;
  try { start = start || localStorage.getItem('fa-ticker'); } catch {}
  start = start || markets.find(m => m.state === 'open')?.tickers[0].symbol;
  if (start) select(start);
  setInterval(loadMarkets, 60000);
})();
</script>
</body>
</html>

In [ ]:
import sys, torch, numpy as np, pandas as pd
sys.path.insert(0, "/content/lab")
for name in [m for m in sys.modules if m == "app" or m.startswith("riskmc")]:
    del sys.modules[name]                      # re-running in the same session: load the files just written
from riskmc import training as T
from riskmc.forecasters import build
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: switch the runtime to T4 GPU for speed)")

## 2. Download training data
About 4 weeks of 1-minute prices per ticker (Yahoo Finance keeps 1-minute bars for ~30 days). Each ticker's history is split by time:

`[ fine-tune (older weeks) ][ calibrate: 1 session ][ test: last 2 sessions ]`

In [ ]:
DATA = T.download_all(T.TRAIN_TICKERS, days=28)
print(f"\n{len(DATA)} tickers, {sum(len(s) for s in DATA.values()):,} one-minute prices")

## 3. Load the models (out of the box)

In [ ]:
MODELS = build(device=DEVICE)
for m in MODELS.values():
    print(f"{m.name:<12} {'loaded' if m.available else 'FAILED: ' + str(m.error)}")

## 4. Score the untrained models on the test sessions
Out of the box, the way most tutorials use these models: they read the last 512 **prices** and forecast the next price.
This is the baseline everything else has to beat.

In [ ]:
N_TEST = 150   # forecasts per ticker
for key in ("chronos2", "timesfm"):
    if MODELS[key].available:
        MODELS[key].input_mode, MODELS[key].context = "prices", 512
ZERO_SHOT = T.summarize(MODELS, T.collect(MODELS, DATA, "test", N_TEST))
ZERO_SHOT.style.format(precision=3)

## 4b. Choose each model's input: prices or returns
Fed raw prices, both models rescale the window by the price level, which drifts far more over 512 minutes than the
next minute moves, so they mostly learn "next price ≈ last price" and fine-tuning has little to work with.
Fed **1-minute returns**, each window is rescaled by its own recent volatility, so the forecast range adapts to calm and
busy markets, the same idea behind the Monte Carlo model. Studies of these models on financial data recommend
returns ([1](https://arxiv.org/html/2511.18578v1), [2](https://arxiv.org/html/2606.27100)).

Rather than assume it, this cell tries prices vs returns and a short vs long context on the **calibration session**
(never the test sessions) and keeps the best setting for each model.

In [ ]:
CONFIG_TABLES = {}
for key in ("chronos2", "timesfm"):
    if MODELS[key].available:
        CONFIG_TABLES[key] = T.select_config(key, MODELS[key], DATA)
        print(MODELS[key].name); display(CONFIG_TABLES[key].style.format(precision=3))
INPUT_FIXED = T.summarize(MODELS, T.collect(MODELS, DATA, "test", N_TEST, verbose=False))
print("Test sessions with the chosen input (still untrained):"); display(INPUT_FIXED.style.format(precision=3))

## 5. Fine-tune Chronos-2
Uses chronos-forecasting's built-in `fit()` with LoRA adapters on the older weeks of every ticker.

**Reading the table:** *Validation Loss* is measured on held-out minutes from the calibration session (never trained on),
starting before step 1. It should fall below its starting value; the checkpoint with the lowest validation loss is kept.
One-minute prices are mostly noise, so expect small gains (a few percent), not a steep curve.

In [ ]:
def training_data(m):
    """Training series, gap-free targets and validation windows in the model's chosen input."""
    if m.input_mode == "returns":       # gap returns are already dropped, so every position is a valid target
        return (T.finetune_series(DATA, mode="returns"), None,
                T.validation_windows(DATA, context=m.context, mode="returns"))
    return (T.finetune_series(DATA, split_sessions=True), None,
            T.validation_windows(DATA, context=m.context))

if MODELS["chronos2"].available:
    m = MODELS["chronos2"]
    series, _, val = training_data(m)
    print(f"Chronos-2: {m.input_mode} input, context {m.context} · {len(series)} training series · {len(val)} validation windows")
    m.fine_tune(series, out_dir="/content/models/chronos2-lora", validation=val, num_steps=600, learning_rate=1e-4)

## 6. Fine-tune TimesFM 2.5
LoRA fine-tuning following TimesFM's own example: random 512-minute windows, predict the next minute,
minimizing the model's built-in quantile loss. Targets right after an overnight or lunch gap are skipped: an opening
jump isn't a 1-minute move, and a few of them can swamp training. Validation loss on the same held-out windows is checked every 25 steps
(and before training); the best adapter is kept.

In [ ]:
if MODELS["timesfm"].available:
    m = MODELS["timesfm"]
    if m.input_mode == "returns":
        series, targets, val = T.finetune_series(DATA, mode="returns"), None, T.validation_windows(DATA, context=m.context, mode="returns")
    else:
        series, targets, val = T.finetune_series(DATA), T.finetune_targets(DATA, m.context), T.validation_windows(DATA, context=m.context)
    print(f"TimesFM: {m.input_mode} input, context {m.context} · {len(val)} validation windows")
    HIST = pd.DataFrame(m.fine_tune(series, out_dir="/content/models/timesfm-lora", validation=val, targets=targets,
                                    num_steps=600, learning_rate=5e-5, batch_size=64))
    ax = HIST.set_index("step")[["train", "val"]].rename(columns={"train": "training", "val": "validation (held out)"}).plot(
        marker="o", ms=3, figsize=(7, 3), title="TimesFM fine-tuning loss")
    ax.axhline(HIST["val"].iloc[0], ls="--", lw=1, color="gray", label="before training"); ax.legend();

## 7. Tune every model's range width
One held-out session is used to fit a single spread multiplier per model (Monte Carlo included),
so all three get the same calibration treatment.

In [ ]:
for m in MODELS.values():
    m.scale = 1.0
SCALES = T.calibrate(MODELS, T.collect(MODELS, DATA, "calib", N_TEST, verbose=False))

## 8. Score the trained models on the test sessions

In [ ]:
TEST_ROWS = T.collect(MODELS, DATA, "test", N_TEST, verbose=False)
TRAINED = T.summarize(MODELS, TEST_ROWS)
print("1. Out of the box (price input)"); display(ZERO_SHOT.style.format(precision=3))
print("2. Chosen input, untrained");      display(INPUT_FIXED.style.format(precision=3))
print("3. Fine-tuned + range width tuned"); display(TRAINED.style.format(precision=3))

In [ ]:
import matplotlib.pyplot as plt
from riskmc.minute import score
by_market = (TEST_ROWS.assign(market=TEST_ROWS.symbol.map({s: m["name"] for m in T.MARKETS for s, _ in m["tickers"]}))
             .groupby(["market", "model"]).apply(lambda r: score(r)["pinball_bps"]).unstack())
by_market = by_market.rename(columns={k: m.name for k, m in MODELS.items()})
ax = by_market.plot.barh(figsize=(8, 7), color=["#eb6834", "#2a78d6", "#1baf7a"][:by_market.shape[1]])
ax.set_xlabel("Pinball loss, bps (lower is better)"); ax.set_title("Test sessions, by market"); plt.tight_layout()

In [ ]:
REPORT = T.save_report("/content/models/report.json", MODELS, ZERO_SHOT, TRAINED, SCALES, DATA,
                       extra={"device": DEVICE, "input_fixed": INPUT_FIXED.reset_index().to_dict(orient="records"),
                              "config_search": {k: t.to_dict(orient="records") for k, t in CONFIG_TABLES.items()}})
print("Saved /content/models/report.json")

## 9. Download the trained models
Saves `trained_models.zip` to your computer: both LoRA adapters (a few MB each) and the range widths.
Unzip it into the project's `models/` folder and run `python app.py` locally (see the README).

In [ ]:
import shutil
from pathlib import Path
pkg = Path("/content/trained_models")
shutil.rmtree(pkg, ignore_errors=True); pkg.mkdir()
shutil.copy("/content/models/report.json", pkg / "report.json")
if Path("/content/models/chronos2-lora/finetuned-ckpt").exists():
    shutil.copytree("/content/models/chronos2-lora/finetuned-ckpt", pkg / "chronos2-lora")
if Path("/content/models/timesfm-lora").exists():
    shutil.copytree("/content/models/timesfm-lora", pkg / "timesfm-lora")
shutil.make_archive("/content/trained_models", "zip", pkg)
!du -sh /content/trained_models.zip
from google.colab import files
files.download("/content/trained_models.zip")

## 10. Live comparison app
Opens the app in a new tab. Pick a ticker from a market that is open now; every minute all three models
lock in a forecast, and each is checked when the real price arrives.

In [ ]:
import app
app.serve(MODELS, REPORT, port=5050, background=True)
from google.colab import output
output.serve_kernel_port_as_window(5050, anchor_text="Open Forecast Arena")

In [ ]:
# Or show it inside the notebook:
output.serve_kernel_port_as_iframe(5050, height=1400)